In [1]:
import re
import numpy as np
import pandas as pd

R = 1.987e-3  # kcal/(mol·K)

def parse_temperature(val):
    """Extract first number from a string, return NaN if none found."""
    match = re.search(r"[\d.]+", str(val))
    return float(match.group()) if match else np.nan

def calc_delta_g(row):
    kd = row["KD(M)"]
    T = parse_temperature(row["Temperature(K)"])

    if np.isnan(T):
        T = 300
    if not (250 <= T <= 350):
        print(f"Error: Temperature {T} K out of range for index {row.name}")
        return np.nan

    return R * T * np.log(kd)  # ΔG = RT·ln(Kd), in kcal/mol



In [2]:
affinities = pd.read_excel( "/FastHome/gyula/PPB-Affinity_two_chain_with_sequences.xlsx")



In [3]:
affinities

,Unnamed: 0,Source Data Set,Complex ID,PDB,Mutations,Ligand Chains,Receptor Chains,Ligand Name,Receptor Name,KD(M),...,Affinity PubMed ID,Affinity Release Date,Subgroup,Chain A (Ligand) ID,Chain B (Receptor) ID,Chain A Sequence (Ligand),Chain B Sequence (Receptor),Chain A Length,Chain B Length,Shorter Than 50 aa
0,0,SKEMPI v2.0,"1A22:A, B::PMID=7504735",1A22,NaN,A,B,Human growth hormone,hGH binding protein,9.000000e-10,...,7504735,1993 Dec 5,NaN,A,B,FPTIPLSRLFDNAMLRAHRLHQLAFDTYQEFEEAYIPKEQKYSFLQ...,FSGSEATAAILSRAPWSLQSVNPGLKTNSSKEPKFTKCRSPERETF...,191,238,False
1,1,SKEMPI v2.0,"1A4Y:A, B::PMID=9050852",1A4Y,NaN,A,B,Ribonuclease inhibitor,Angiogenin,5.000000e-16,...,9050852,1997 Mar 4,NaN,A,B,SLDIQSLDIQCEELSDARWAELLPLLQQCQVVRLDDCGLTEARCKD...,QDNSRYTHFLTQHYDAKPQGRDDRYCESIMRRRGLTSPCKDINTFI...,460,123,False
2,2,SKEMPI v2.0,"1ACB:E, I::PMID=9048543",1ACB,NaN,E,I,Bovine alpha-chymotrypsin,Eglin c,1.490000e-12,...,9048543,1997 Feb 18,NaN,E,I,CGVPAIQPVLSGLSRIVNGEEAVPGSWPWQVSLQDKTGFHFCGGSL...,TEFGSELKSFPEVVGKTVDQAREYFTLHYPQYDVYFLPEGSPVTLD...,245,70,False
3,4,SKEMPI v2.0,"1AK4:A, D::PMID=9223641",1AK4,NaN,A,D,Cyclophilin A,HIV-1 capsid protein,1.200000e-05,...,9223641,1997 Jun 27,NaN,A,D,MVNPTVFFDIAVDGEPLGRVSFELFADKVPKTAENFRALSTGEKGF...,PIVQNLQGQMVHQAISPRTLNAWVKVVEEKAFSPEVIPMFSALSEG...,165,145,False
4,6,SKEMPI v2.0,"1B2S:A, D::PMID=7739054",1B2S,NaN,A,D,Barnase,Barstar,1.570000e-10,...,7739054,1995 Apr 28,NaN,A,D,AQVINTFDGVADYLQTYHKLPDNYITASEAQALGWVASKGNLADVA...,MKKAVINGEQIRSISDLHQTLKKELALPEYYGENLDALWDCLAGWV...,110,90,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6431,12391,PDBbind v2020,"6THG:I, J::PMID=31862858",6THG,NaN,J,I,human ephrin-B1,"Cedar Virus attachment glycoprotein (G), CedVG",4.000000e-09,...,31862858,2020 Jan,NaN,J,I,ETGAKNLEPVSWSSLNPKFLSGKGLVIYPKIGDKLDIICPRAEAGR...,ETGKIFCKSVSKDPDFRLKQIDYVIPVQQDRSICMNNPLLDISDGF...,151,426,False
6432,12402,PDBbind v2020,"6UMT:A, B::PMID=31727844",6UMT,NaN,B,A,"Programmed cell death 1 ligand 2, PD-L2 IgV","Programmed cell death protein 1, human PD-1 (N...",2.600000e-09,...,31727844,2019 Dec 3,NaN,B,A,MIFLLLMLSLELQLHQIAALFTVTVPKELYIIEHGSDVTLECNFDT...,MGWSCIILFLVATATGVHSNPPTFSPALLVVTEGDSATFTCSFSST...,123,140,False
6433,12404,PDBbind v2020,"6UYS:A, B::PMID=31879127",6UYS,NaN,B,A,phosphorylated PML-SIM,K37-acetylated SUMO1,1.600000e-06,...,31879127,2020 Feb 4,NaN,B,A,GSGAGEAEERVVVISSSEDSDAENSSSRY,GSKEGEYIKLKVIGQDSSEIHFKVKMTTHLKKLKESYAQRQGVPMN...,29,83,True
6434,12405,PDBbind v2020,"6UYS:C, D::PMID=31879127",6UYS,NaN,D,C,phosphorylated PML-SIM,K37-acetylated SUMO1,1.600000e-06,...,31879127,2020 Feb 4,NaN,D,C,GSGAGEAEERVVVISSSEDSDAENSSSRY,GSKEGEYIKLKVIGQDSSEIHFKVKMTTHLKKLKESYAQRQGVPMN...,29,83,True


In [4]:
# Calculate deltaGs from KDs
affinities["Y"] = affinities.apply(calc_delta_g, axis=1)

In [5]:
swap_pairs = [
    ("Chain A Length",            "Chain B Length"),
    ("Ligand Name",               "Receptor Name"),
    ("Ligand Chains",             "Receptor Chains"),
    ("Chain A (Ligand) ID",       "Chain B (Receptor) ID"),
    ("Chain A Sequence (Ligand)", "Chain B Sequence (Receptor)"),
]

# Build a copy with the columns swapped
swapped = affinities.copy()
for col_a, col_b in swap_pairs:
    swapped[[col_a, col_b]] = affinities[[col_b, col_a]].values

# Concatenate original + swapped rows
affinities_augmented = pd.concat(
    [affinities, swapped], ignore_index=True
)

In [6]:
affinities_augmented["Target"] = affinities_augmented["Chain A Sequence (Ligand)"]

affinities_augmented["proteina"] = affinities_augmented["Chain B Sequence (Receptor)"]

In [7]:
affinities_augmented.to_csv("/FastHome/gyula/ppi_data/train_sets/PPI_augmented_ppi_experimental.csv")

In [8]:
max(affinities_augmented.Y)

-1.8986243695992426

In [9]:

df = affinities_augmented
print("raw:", len(df))
for c in ["Y", "Target", "proteina", "PDB", "Subgroup"]:
    print(f"{c}: {df[c].isna().sum()} NaN")

raw: 12872
Y: 0 NaN
Target: 2 NaN
proteina: 2 NaN
PDB: 0 NaN
Subgroup: 12340 NaN


<bound method NDFrame.head of       Unnamed: 0 Source Data Set                Complex ID   PDB Mutations  \
0              0     SKEMPI v2.0   1A22:A, B::PMID=7504735  1A22       NaN   
1              1     SKEMPI v2.0   1A4Y:A, B::PMID=9050852  1A4Y       NaN   
2              2     SKEMPI v2.0   1ACB:E, I::PMID=9048543  1ACB       NaN   
3              4     SKEMPI v2.0   1AK4:A, D::PMID=9223641  1AK4       NaN   
4              6     SKEMPI v2.0   1B2S:A, D::PMID=7739054  1B2S       NaN   
...          ...             ...                       ...   ...       ...   
6431       12391   PDBbind v2020  6THG:I, J::PMID=31862858  6THG       NaN   
6432       12402   PDBbind v2020  6UMT:A, B::PMID=31727844  6UMT       NaN   
6433       12404   PDBbind v2020  6UYS:A, B::PMID=31879127  6UYS       NaN   
6434       12405   PDBbind v2020  6UYS:C, D::PMID=31879127  6UYS       NaN   
6435       12406   PDBbind v2020  6UYU:A, B::PMID=31879127  6UYU       NaN   

     Ligand Chains Receptor Chain

In [12]:
aff = pd.read_excel( "/FastHome/gyula/PPB-Affinity.xlsx")

In [18]:
aff

,Unnamed: 0,Source Data Set,Complex ID,PDB,Mutations,Ligand Chains,Receptor Chains,Ligand Name,Receptor Name,KD(M),Affinity Method,Structure Method,Temperature(K),Resolution(Å),PDB PubMed ID,PDB Release Date,Affinity PubMed ID,Affinity Release Date,Subgroup
0,0,SKEMPI v2.0,"1A22:A, B::PMID=7504735",1A22,NaN,A,B,Human growth hormone,hGH binding protein,9.000000e-10,SPR,X-RAY DIFFRACTION,298,2.60,9571026.0,1998-04-29,7504735,1993 Dec 5,NaN
1,1,SKEMPI v2.0,"1A4Y:A, B::PMID=9050852",1A4Y,NaN,A,B,Ribonuclease inhibitor,Angiogenin,5.000000e-16,Other,X-RAY DIFFRACTION,298,2.00,9311977.0,1998-10-14,9050852,1997 Mar 4,NaN
2,2,SKEMPI v2.0,"1ACB:E, I::PMID=9048543",1ACB,NaN,E,I,Bovine alpha-chymotrypsin,Eglin c,1.490000e-12,IASP,X-RAY DIFFRACTION,294,2.00,1583684.0,1993-10-31,9048543,1997 Feb 18,NaN
3,3,SKEMPI v2.0,"1AHW:A, B, C::PMID=9480775",1AHW,NaN,"A, B",C,Immunoglobulin fab 5G9,Tissue factor,3.400000e-09,IASP,X-RAY DIFFRACTION,298(assumed),3.00,9480775.0,1998-02-25,9480775,1998 Feb 6,NaN
4,4,SKEMPI v2.0,"1AK4:A, D::PMID=9223641",1AK4,NaN,A,D,Cyclophilin A,HIV-1 capsid protein,1.200000e-05,SPR,X-RAY DIFFRACTION,298(assumed),2.36,8980234.0,1997-10-15,9223641,1997 Jun 27,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12057,13039,ATLAS,"4L3E:A, C, D, E:A_E166A:PMID=23736024",4L3E,A_E166A,"A, C","D, E",ELAGIGILTV-HLA-A*02:01,DMF5,3.220000e-05,SPR,X-RAY DIFFRACTION,298.15,2.56,24550723.0,2014-06-11,23736024,2013-01-01,TCR-pMHC
12058,13040,ATLAS,"4L3E:A, C, D, E:A_Q155A, D_Y50A:PMID=23736024",4L3E,"A_Q155A, D_Y50A","A, C","D, E",ELAGIGILTV-HLA-A*02:01,DMF5,7.710000e-05,SPR,X-RAY DIFFRACTION,298.15,2.56,24550723.0,2014-06-11,23736024,2013-01-01,TCR-pMHC
12059,13041,ATLAS,"4L3E:A, C, D, E:A_E166A, D_N52A:PMID=23736024",4L3E,"A_E166A, D_N52A","A, C","D, E",ELAGIGILTV-HLA-A*02:01,DMF5,2.610000e-05,SPR,X-RAY DIFFRACTION,298.15,2.56,24550723.0,2014-06-11,23736024,2013-01-01,TCR-pMHC
12060,13042,ATLAS,"4L3E:A, C, D, E:D_D26Y, E_L98W:PMID=22611242",4L3E,"D_D26Y, E_L98W","A, C","D, E",ELAGIGILTV-HLA-A*02:01,DMF5,4.300000e-08,SPR,X-RAY DIFFRACTION,298.15,2.56,24550723.0,2014-06-11,22611242,2012-06-15,TCR-pMHC


In [1]:
aff.loc[:,"Y Method"].min()

NameError: name 'aff' is not defined